<a href="https://colab.research.google.com/github/HLZHarry/AlphaLink-Muti-Modal-RAG/blob/main/Step_2_The_File_Reorganzier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from pathlib import Path
import shutil
import os

In [23]:
# 1. Setup paths using Pathlib
base_path = Path("/content/drive/MyDrive/AlphaLink-RAG/data")
raw_dir = base_path / "raw" / "sec-edgar-filings"
clean_dir = base_path / "processed_filings"

In [31]:
# Create the folder if it doesn't exist
clean_dir.mkdir(parents=True, exist_ok=True)

print(f"🚀 Moving files from {raw_dir} to {clean_dir}...")

count = 0
# rglob("*") finds every file in all subfolders recursively
for file_path in raw_dir.rglob("*"):
    # Only process actual filing files (skip folders and tiny metadata)
    if file_path.is_file() and file_path.stat().st_size > 5000:

        try:
            # Extract naming info: Ticker is 4 levels up, Form is 3 levels up
            ticker = file_path.parts[-4]
            form = file_path.parts[-3]
            accession = file_path.parts[-2]

            new_name = f"{ticker}_{form}_{accession}{file_path.suffix}"

            # Use copy2 to preserve metadata (like filing dates)
            shutil.copy2(file_path, clean_dir / new_name)
            count += 1
        except Exception as e:
            continue

print(f"✨ SUCCESS! {count} files are now permanently in: {clean_dir}")
print("Check your Google Drive web tab now—the 'processed_filings' folder should be there!")

🚀 Moving files from /content/drive/MyDrive/AlphaLink-RAG/data/raw/sec-edgar-filings to /content/drive/MyDrive/AlphaLink-RAG/data/processed_filings...
✨ SUCCESS! 435 files are now permanently in: /content/drive/MyDrive/AlphaLink-RAG/data/processed_filings
Check your Google Drive web tab now—the 'processed_filings' folder should be there!


- **`parents=True`** — Tells Python: *"If the middle folders are missing (like `/data/`), create those too."*

- **`exist_ok=True`** — The **"Safety Switch."** Prevents the script from crashing if the folder already exists.

- **`rglob("*")`** — A **Recursive Glob.** Tells Python to dive into every single subfolder inside `raw_dir` and find every file (`*`). Think of it as a high-speed search through a filing cabinet.

- **`is_file()`** — Ignores folders and only processes actual documents.

- **`st_size > 5000`** — Ignores tiny "junk" files (under 5KB) that are often just empty headers or metadata.

- **`.parts`** — The most clever piece. Splits the file path into a list of names. By counting backwards from the end (`-4`, `-3`), we can grab the **Ticker** and **Form type** regardless of how deep they were buried by the SEC downloader.

- **`shutil.copy2`** — The `2` version preserves the original file **metadata** (like the filing date). This is critical for our RAG — we want the AI to know exactly *when* NVIDIA said they were low on chips.